# 05. Dashboard Data Export

**Purpose:** Run every scenario through the model and export one tidy table for Power BI.

**Output:** `data/clean/dashboard_scenarios.csv` (one row per scenario, city, industry and housing type)

## run all scenarios

In [1]:
from model_utils import *
import pandas as pd

tables = load_tables()

SCENARIOS = {
    "Base": {},
    "Rent +10%": {"rent_shock": 0.10},
    "12 hours/week": {"hours": scale_hours(12)},
    "48 hours/week (hypothetical)": {"hours": scale_hours(48)},
    "Foreign resident": {"residency": "Foreign resident"},
    "Food low ($49)": {"food": 49.0},
    "Food high ($150)": {"food": 150.0},
}

rows = []
for name, kw in SCENARIOS.items():
    for city in CITIES:
        for industry in AWARD:
            for housing in HOUSING:
                r = disposable_income(tables, city, industry=industry, housing=housing, **kw)
                rows.append({
                    "scenario": name,
                    "city": city,
                    "industry": industry,
                    "housing": housing,
                    "gross_weekly": round(r["gross"], 2),
                    "net_weekly": round(r["net"], 2),
                    "essentials_weekly": round(r["essentials"], 2),
                    "disposable_weekly": round(r["disposable"], 2),
                })

scen = pd.DataFrame(rows)
print(scen.shape)   # expect (140, 8)
scen.head()

(140, 8)


,scenario,city,industry,housing,gross_weekly,net_weekly,essentials_weekly,disposable_weekly
0,Base,Sydney,Hospitality,Share room,872.52,799.21,769.60,29.61
1,Base,Sydney,Hospitality,Living alone,872.52,799.21,980.19,-180.98
2,Base,Sydney,Retail,Share room,959.46,858.77,769.60,89.16
3,Base,Sydney,Retail,Living alone,959.46,858.77,980.19,-121.43
4,Base,Melbourne,Hospitality,Share room,872.52,799.21,604.10,195.11


In [2]:
# Base scenario must match the model table
base = scen[scen["scenario"] == "Base"].set_index(["city", "industry", "housing"])["disposable_weekly"]
print("Base share room, Hospitality:")
print(base.xs(("Hospitality", "Share room"), level=(1, 2)).round(2).to_dict())

assert scen.notna().all().all()
scen.to_csv("../data/clean/dashboard_scenarios.csv", index=False)
print("Saved dashboard_scenarios.csv")

Base share room, Hospitality:
{'Sydney': 29.61, 'Melbourne': 195.11, 'Brisbane': 169.61, 'Perth': 168.21, 'Adelaide': 200.21}
Saved dashboard_scenarios.csv
